# 📖 Notebook 1: Setup + IndicQA Evaluation

**Indic Language Benchmark Suite**

This notebook:
1. Installs dependencies
2. Loads Sarvam-2B, Gemma-2B, and Llama-3.2-1B (4-bit quantized)
3. Loads IndicQA dataset for Hindi, Bengali, and Tamil
4. Runs reading comprehension inference on all models
5. Computes Exact Match and F1 metrics
6. Saves intermediate results

**Runtime:** Set to GPU (T4) via Runtime → Change runtime type

## 1. Setup & Install Dependencies

In [ ]:
# Clone the repo (run once)
# !git clone https://github.com/YOUR_USERNAME/lang-benchmark.git
# %cd lang-benchmark

# Or if running locally, set the path
import os
# os.chdir('/content/lang-benchmark')  # Uncomment for Colab

In [ ]:
%%capture
!pip install -q transformers>=4.40.0 torch accelerate bitsandbytes
!pip install -q datasets sentencepiece protobuf evaluate
!pip install -q rouge-score nltk pandas tabulate tqdm jsonlines
!pip install -q matplotlib seaborn huggingface_hub

In [ ]:
# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# HuggingFace login (needed for Llama and Gemma gated models)
from huggingface_hub import notebook_login
notebook_login()

## 2. Import Benchmark Modules

In [ ]:
# Add src to path if running from notebooks/
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.getcwd())

from src.data_loader import load_indicqa, load_code_mixed_qa
from src.model_loader import (
    load_model_and_tokenizer, unload_model,
    compute_fertility, get_model_info
)
from src.inference import run_inference, build_prompt
from src.metrics import (
    evaluate_results, format_metrics_report,
    exact_match, f1_score
)

print(get_model_info())

## 3. Load IndicQA Dataset

In [ ]:
# Load IndicQA for Hindi, Bengali, and Tamil
# Using a smaller sample size for faster iteration; increase for final run
MAX_SAMPLES = 100  # Set to 500 for the final benchmark run

indicqa_samples = load_indicqa(
    languages=["hindi", "bengali", "tamil"],
    max_samples_per_lang=MAX_SAMPLES,
    split="test",
)

In [ ]:
# Inspect a few samples
import pandas as pd

df = pd.DataFrame(indicqa_samples)
print(f"Total samples: {len(df)}")
print(f"\nLanguage distribution:")
print(df['language'].value_counts())
print(f"\nSample entry:")
print(df.iloc[0].to_dict())

## 4. Evaluate Each Model on IndicQA

We load one model at a time, run inference, then unload to free GPU memory.

**Order:** Sarvam-2B → Gemma-2B → Llama-3.2-1B

In [ ]:
# Configuration
MODELS_TO_EVAL = ["sarvam-2b", "gemma-2b", "llama-3.2-1b"]
TASK = "reading_comprehension"
RESULTS_DIR = "results/raw"
os.makedirs(RESULTS_DIR, exist_ok=True)

all_model_results = {}

In [ ]:
# ─── Model 1: Sarvam-2B ───────────────────────────────────────
model, tokenizer, config = load_model_and_tokenizer("sarvam-2b", quantize=True)

sarvam_results = run_inference(
    model=model,
    tokenizer=tokenizer,
    samples=indicqa_samples,
    task=TASK,
    model_name=config.name,
    max_new_tokens=64,
    use_fewshot=True,
    save_path=f"{RESULTS_DIR}/sarvam-2b_reading_comprehension.jsonl",
)

all_model_results["sarvam-2b"] = sarvam_results

# Evaluate
sarvam_metrics = evaluate_results(sarvam_results, TASK)
print(format_metrics_report(sarvam_metrics, config.name))

# Unload to free memory
unload_model(model, tokenizer)

In [ ]:
# ─── Model 2: Gemma-2B ────────────────────────────────────────
model, tokenizer, config = load_model_and_tokenizer("gemma-2b", quantize=True)

gemma_results = run_inference(
    model=model,
    tokenizer=tokenizer,
    samples=indicqa_samples,
    task=TASK,
    model_name=config.name,
    max_new_tokens=64,
    use_fewshot=True,
    save_path=f"{RESULTS_DIR}/gemma-2b_reading_comprehension.jsonl",
)

all_model_results["gemma-2b"] = gemma_results

# Evaluate
gemma_metrics = evaluate_results(gemma_results, TASK)
print(format_metrics_report(gemma_metrics, config.name))

# Unload
unload_model(model, tokenizer)

In [ ]:
# ─── Model 3: Llama-3.2-1B ───────────────────────────────────
model, tokenizer, config = load_model_and_tokenizer("llama-3.2-1b", quantize=True)

llama_results = run_inference(
    model=model,
    tokenizer=tokenizer,
    samples=indicqa_samples,
    task=TASK,
    model_name=config.name,
    max_new_tokens=64,
    use_fewshot=True,
    save_path=f"{RESULTS_DIR}/llama-3.2-1b_reading_comprehension.jsonl",
)

all_model_results["llama-3.2-1b"] = llama_results

# Evaluate
llama_metrics = evaluate_results(llama_results, TASK)
print(format_metrics_report(llama_metrics, config.name))

# Unload
unload_model(model, tokenizer)

## 5. Compare Results — Reading Comprehension

In [ ]:
# Side-by-side comparison
print("\n" + "=" * 70)
print("📊 READING COMPREHENSION — COMPARISON TABLE")
print("=" * 70)

comparison_data = []
for model_key, results in all_model_results.items():
    metrics = evaluate_results(results, TASK)
    em = metrics.get('exact_match', {}).get('exact_match', 0)
    f1 = metrics.get('f1', {}).get('f1', 0)
    
    row = {'Model': model_key, 'EM (%)': em, 'F1 (%)': f1}
    
    # Per-language breakdown
    for lang in ['hindi', 'bengali', 'tamil']:
        lang_em = metrics.get('exact_match', {}).get('per_language', {}).get(lang, 0)
        lang_f1 = metrics.get('f1', {}).get('per_language', {}).get(lang, 0)
        row[f'{lang.title()} EM'] = lang_em
        row[f'{lang.title()} F1'] = lang_f1
    
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_markdown(index=False))

In [ ]:
# Spot-check: view some predictions
print("\n📝 Sample Predictions (first 5):")
print("-" * 70)

for model_key, results in all_model_results.items():
    print(f"\n🤖 {model_key}:")
    for r in results[:5]:
        print(f"  Q: {r['question'][:80]}...")
        print(f"  Gold: {r['gold_answer']}")
        print(f"  Pred: {r['prediction'][:100]}")
        print(f"  EM: {exact_match(r['prediction'], r['gold_answer'])}")
        print()

## 6. Save Checkpoint

All raw results are saved as JSONL files in `results/raw/`.
These will be used in Notebook 2 for the combined leaderboard.

In [ ]:
# Verify saved files
print("📂 Saved result files:")
for f in os.listdir(RESULTS_DIR):
    filepath = os.path.join(RESULTS_DIR, f)
    size = os.path.getsize(filepath)
    print(f"  {f} ({size / 1024:.1f} KB)")

print("\n✅ Day 1 complete! Proceed to Notebook 2 for math & code-mixed eval.")